## 📚 Comprehensive LlamaIndex Tutorial

This notebook provides an in-depth guide to **LlamaIndex**, a powerful framework for building data-augmented AI applications with Large Language Models (LLMs). It extends the original `Llama_Index_basic.ipynb` by preserving its code, adding theoretical explanations, detailed comments, and introducing advanced functionalities not covered in the original.

## What is LlamaIndex?
LlamaIndex is a data framework that connects LLMs with external data sources for Retrieval-Augmented Generation (RAG) and other applications. It simplifies indexing, querying, and interacting with structured and unstructured data. Key features include:
- **Indexing**: Organizes data for efficient retrieval (e.g., vector, keyword, summary indexes).
- **Query Engines**: Supports semantic, keyword, and hybrid search.
- **Chat Engines**: Enables conversational interfaces with context.
- **Agents**: Integrates tools for dynamic interactions.
- **Evaluation**: Assesses response quality (e.g., faithfulness, relevance).

## Objectives
- Demonstrate all major LlamaIndex functionalities from the original notebook.
- Introduce advanced features like document summary indexes, custom retrievers, sub-question query engines, SQL integration, and embedding fine-tuning.
- Provide theoretical context and detailed comments.
- Preserve and enhance the original code for clarity and reusability.

## Prerequisites
- Python 3.10+
- Install dependencies: `pip install llama-index llama-index-llms-ollama llama-index-embeddings-huggingface llama-index-vector-stores-chroma chromadb sqlalchemy`
- Local Ollama server running with `llama3` model.
- Sample data directory (`data/`) with text files.
- SQLite database for SQL integration example.

## Structure
1. **Setup and Basic Indexing**
2. **Vector Store and Query Engine**
3. **Chat Engine**
4. **Persistent Storage with Chroma**
5. **Node Parsing**
6. **Hybrid Search with Router Query Engine**
7. **Agent with Tools**
8. **Evaluation**
9. **Document Summary Index**
10. **Custom Retrievers**
11. **Sub-Question Query Engine**
12. **SQL Database Integration**
13. **Fine-Tuning Embeddings**

Let's dive in!

## 1. Setup and Basic Indexing

**Theory**: LlamaIndex requires initializing an LLM, embeddings, and document loaders to process data. The original notebook uses Ollama's `llama3` model and HuggingFace embeddings for open-source processing. This section sets up the environment and loads documents.

In [ ]:
# Install required packages
!pip install llama-index llama-index-llms-ollama llama-index-embeddings-huggingface llama-index-vector-stores-chroma chromadb sqlalchemy

In [ ]:
# Original Basic Indexing
from llama_index.core import SimpleDirectoryReader
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings

# Set up LLM and embeddings
# Theory: Settings globally configures the LLM and embeddings for all LlamaIndex operations.
Settings.llm = Ollama(model="llama3")
Settings.embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Load documents
# Theory: SimpleDirectoryReader loads text files from a directory, creating Document objects.
documents = SimpleDirectoryReader(input_dir="data/").load_data()

# Print loaded documents
for doc in documents:
    print("Document:", doc.text[:100])

## 2. Vector Store and Query Engine

**Theory**: LlamaIndex's `VectorStoreIndex` creates a vector store for semantic search, embedding documents and enabling similarity-based retrieval. The original notebook demonstrates creating and querying a vector index. We'll preserve this with added comments.

In [ ]:
# Original Vector Store and Query Engine
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings

# Set up LLM and embeddings
Settings.llm = Ollama(model="llama3")
Settings.embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Load documents
documents = SimpleDirectoryReader(input_dir="data/").load_data()

# Create vector index
# Theory: VectorStoreIndex embeds documents into a vector space for efficient semantic search.
index = VectorStoreIndex.from_documents(documents)

# Print index stats
print("Index created with", len(index.docstore.docs), "documents")

# Create query engine
# Theory: The query engine retrieves top-k similar documents and generates answers using the LLM.
query_engine = index.as_query_engine(similarity_top_k=2)

# Query
response = query_engine.query("What does LlamaIndex support?")
print("Answer:", response)

## 3. Chat Engine

**Theory**: LlamaIndex's chat engine supports conversational interfaces by maintaining context and condensing follow-up questions. The original notebook uses the `condense_question` mode. We'll preserve this example with enhanced comments.

In [ ]:
# Original Chat Engine
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings

# Set up LLM and embeddings
Settings.llm = Ollama(model="llama3")
Settings.embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Load documents and create index
documents = SimpleDirectoryReader(input_dir="data/").load_data()
index = VectorStoreIndex.from_documents(documents)

# Create chat engine
# Theory: 'condense_question' mode rewrites follow-up questions to be standalone, preserving context.
chat_engine = index.as_chat_engine(chat_mode="condense_question", verbose=True)

# Simulate conversation
responses = [
    chat_engine.chat("What is LlamaIndex?"),
    chat_engine.chat("What does it support?")
]
for i, response in enumerate(responses, 1):
    print(f"Response {i}:", response)

## 4. Persistent Storage with Chroma

**Theory**: LlamaIndex supports persistent storage using vector stores like Chroma, allowing indexes to be saved and reloaded. The original notebook demonstrates this with Chroma. We'll preserve this with added explanations.

In [ ]:
# Original Persistent Storage with Chroma
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, StorageContext
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import Settings
import chromadb

# Set up LLM and embeddings
Settings.llm = Ollama(model="llama3")
Settings.embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Initialize Chroma client
# Theory: Chroma stores embeddings persistently, enabling scalable vector search.
chroma_client = chromadb.PersistentClient(path="./chroma_db")
chroma_collection = chroma_client.create_collection("llama_index", get_or_create=True)

# Load documents
documents = SimpleDirectoryReader(input_dir="data/").load_data()

# Create storage context
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# Create and save index
index = VectorStoreIndex.from_documents(documents, storage_context=storage_context)
index.storage_context.persist(persist_dir="./chroma_db")

# Load index
loaded_index = VectorStoreIndex.from_vector_store(vector_store)
query_engine = loaded_index.as_query_engine()

# Query
response = query_engine.query("What is LlamaIndex?")
print("Answer:", response)

## 5. Node Parsing

**Theory**: LlamaIndex's node parsers split documents into smaller chunks (nodes) for granular indexing. The original notebook uses `SentenceSplitter` for chunking. We'll preserve this with additional comments.

In [ ]:
# Original Node Parsing
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings
from llama_index.core.node_parser import SentenceSplitter

# Set up LLM and embeddings
Settings.llm = Ollama(model="llama3")
Settings.embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Load documents
documents = SimpleDirectoryReader(input_dir="data/").load_data()

# Define node parser
# Theory: SentenceSplitter chunks documents into sentences or fixed-size chunks for better retrieval.
node_parser = SentenceSplitter(chunk_size=200, chunk_overlap=50)

# Parse documents into nodes
nodes = node_parser.get_nodes_from_documents(documents)

# Create index from nodes
index = VectorStoreIndex(nodes)

# Create query engine
query_engine = index.as_query_engine()

# Query
response = query_engine.query("What is LlamaIndex?")
print("Answer:", response)

## 6. Hybrid Search with Router Query Engine

**Theory**: LlamaIndex's `RouterQueryEngine` selects between multiple query engines (e.g., vector and keyword) based on query type. The original notebook demonstrates hybrid search. We'll preserve this with enhanced explanations.

In [ ]:
# Original Hybrid Search with Router Query Engine
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, KeywordTableIndex
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings
from llama_index.core.query_engine import RouterQueryEngine
from llama_index.core.tools import QueryEngineTool

# Set up LLM and embeddings
Settings.llm = Ollama(model="llama3")
Settings.embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Load documents
documents = SimpleDirectoryReader(input_dir="data/").load_data()

# Create vector and keyword indexes
# Theory: VectorStoreIndex enables semantic search; KeywordTableIndex supports exact keyword matching.
vector_index = VectorStoreIndex.from_documents(documents)
keyword_index = KeywordTableIndex.from_documents(documents)

# Create query engines
vector_query_engine = vector_index.as_query_engine(similarity_top_k=2)
keyword_query_engine = keyword_index.as_query_engine()

# Define tools
tools = [
    QueryEngineTool.from_defaults(vector_query_engine, name="vector", description="Semantic search"),
    QueryEngineTool.from_defaults(keyword_query_engine, name="keyword", description="Keyword search")
]

# Create hybrid query engine
# Theory: RouterQueryEngine dynamically selects the best tool based on query content.
query_engine = RouterQueryEngine.from_defaults(query_engine_tools=tools)

# Query
response = query_engine.query("What does LlamaIndex support for search?")
print("Answer:", response)

## 7. Agent with Tools

**Theory**: LlamaIndex's `ReActAgent` integrates tools (e.g., calculator) with LLMs for dynamic task handling. The original notebook includes a calculator tool. We'll preserve this with added comments and fix the noted issue where the agent fails to use the index for LlamaIndex-related queries.

In [ ]:
# Enhanced Agent with Tools
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings
from llama_index.core.tools import FunctionTool, QueryEngineTool
from llama_index.core.agent import ReActAgent

# Set up LLM and embeddings
Settings.llm = Ollama(model="llama3")
Settings.embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Define calculator tool
def calculate(expression: str) -> str:
    """Evaluates a mathematical expression and returns the result."""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {str(e)}"

calc_tool = FunctionTool.from_defaults(fn=calculate, name="calculator")

# Load documents and create index
documents = SimpleDirectoryReader(input_dir="data/").load_data()
index = VectorStoreIndex.from_documents(documents)

# Create query engine tool
# Fix: Add query engine as a tool to enable the agent to retrieve LlamaIndex information.
query_tool = QueryEngineTool.from_defaults(
    index.as_query_engine(similarity_top_k=2),
    name="llama_index_query",
    description="Query the LlamaIndex documentation for information"
)

# Create agent with tools
agent = ReActAgent.from_tools([calc_tool, query_tool], llm=Settings.llm, verbose=True)

# Query
response = agent.chat("What is 5 * 3, and what is LlamaIndex?")
print("Answer:", response)

## 8. Evaluation

**Theory**: LlamaIndex provides evaluators like `FaithfulnessEvaluator` to assess response quality. The original notebook includes this. We'll preserve it with corrections (e.g., ensuring `nest_asyncio` is applied early) and add comments.

In [ ]:
# Original Evaluation with Fix
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings
from llama_index.core.evaluation import FaithfulnessEvaluator
import nest_asyncio

# Apply nest_asyncio to handle async operations in Jupyter
nest_asyncio.apply()

# Set up LLM and embeddings
Settings.llm = Ollama(model="llama3")
Settings.embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Load documents and create index
documents = SimpleDirectoryReader(input_dir="data/").load_data()
index = VectorStoreIndex.from_documents(documents)

# Create query engine
query_engine = index.as_query_engine()

# Define evaluator
# Theory: FaithfulnessEvaluator checks if the response aligns with retrieved documents.
evaluator = FaithfulnessEvaluator(llm=Settings.llm)

# Query and evaluate
query = "What does LlamaIndex support?"
response = query_engine.query(query)
eval_result = evaluator.evaluate_response(query=query, response=response)
print("Answer:", response)
print("Evaluation:", eval_result.passing, eval_result.feedback)

## 9. Document Summary Index

**Theory**: The `DocumentSummaryIndex` generates summaries for documents, enabling high-level retrieval for large datasets. This is useful for quickly understanding document content without retrieving full texts. This feature is not in the original notebook.

In [ ]:
# New Document Summary Index
from llama_index.core import DocumentSummaryIndex, SimpleDirectoryReader
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings
from llama_index.core.node_parser import SentenceSplitter

# Set up LLM and embeddings
Settings.llm = Ollama(model="llama3")
Settings.embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Load documents
documents = SimpleDirectoryReader(input_dir="data/").load_data()

# Define node parser
node_parser = SentenceSplitter(chunk_size=200, chunk_overlap=50)

# Create summary index
# Theory: DocumentSummaryIndex generates summaries for each document, enabling high-level retrieval.
index = DocumentSummaryIndex.from_documents(
    documents,
    node_parser=node_parser,
    llm=Settings.llm
)

# Create query engine
query_engine = index.as_query_engine(response_mode="tree_summarize")

# Query
response = query_engine.query("What is the main purpose of LlamaIndex?")
print("Answer:", response)

## 10. Custom Retrievers

**Theory**: LlamaIndex allows custom retrievers to combine multiple retrieval strategies (e.g., vector and BM25). This provides flexibility for complex retrieval needs. This feature is not in the original notebook.

In [ ]:
# New Custom Retriever
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings
from llama_index.core.retrievers import BaseRetriever, VectorIndexRetriever, BM25Retriever
from llama_index.core import QueryBundle
from typing import List
from llama_index.core.schema import NodeWithScore

# Define custom retriever
class HybridRetriever(BaseRetriever):
    def __init__(self, vector_retriever, bm25_retriever):
        self.vector_retriever = vector_retriever
        self.bm25_retriever = bm25_retriever
        super().__init__()

    def _retrieve(self, query_bundle: QueryBundle) -> List[NodeWithScore]:
        """Combines vector and BM25 retrieval with score-based ranking."""
        vector_nodes = self.vector_retriever.retrieve(query_bundle)
        bm25_nodes = self.bm25_retriever.retrieve(query_bundle)
        all_nodes = vector_nodes + bm25_nodes
        # Remove duplicates based on node ID
        unique_nodes = {node.node.node_id: node for node in all_nodes}
        return sorted(unique_nodes.values(), key=lambda x: x.score, reverse=True)[:2]

# Set up LLM and embeddings
Settings.llm = Ollama(model="llama3")
Settings.embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Load documents
documents = SimpleDirectoryReader(input_dir="data/").load_data()

# Create vector index
vector_index = VectorStoreIndex.from_documents(documents)

# Create retrievers
vector_retriever = VectorIndexRetriever(index=vector_index, similarity_top_k=2)
bm25_retriever = BM25Retriever.from_defaults(nodes=vector_index.docstore.docs.values(), similarity_top_k=2)

# Create hybrid retriever
hybrid_retriever = HybridRetriever(vector_retriever, bm25_retriever)

# Create query engine with custom retriever
query_engine = vector_index.as_query_engine(retriever=hybrid_retriever)

# Query
response = query_engine.query("What does LlamaIndex support?")
print("Answer:", response)

## 11. Sub-Question Query Engine

**Theory**: The `SubQuestionQueryEngine` breaks down complex queries into sub-questions, querying them individually and synthesizing results. This is useful for multi-faceted questions. This feature is not in the original notebook.

In [ ]:
# New Sub-Question Query Engine
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings
from llama_index.core.query_engine import SubQuestionQueryEngine
from llama_index.core.tools import QueryEngineTool

# Set up LLM and embeddings
Settings.llm = Ollama(model="llama3")
Settings.embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Load documents
documents = SimpleDirectoryReader(input_dir="data/").load_data()

# Create vector index
index = VectorStoreIndex.from_documents(documents)

# Create query engine
query_engine = index.as_query_engine(similarity_top_k=2)

# Define tool
query_tool = QueryEngineTool.from_defaults(
    query_engine,
    name="llama_index",
    description="Query LlamaIndex documentation"
)

# Create sub-question query engine
# Theory: SubQuestionQueryEngine decomposes complex queries into sub-questions for detailed answers.
sub_query_engine = SubQuestionQueryEngine.from_defaults(
    query_engine_tools=[query_tool],
    llm=Settings.llm,
    verbose=True
)

# Query
response = sub_query_engine.query("What is LlamaIndex, and what are its key features for data access?")
print("Answer:", response)

## 12. SQL Database Integration

**Theory**: LlamaIndex can query structured data in SQL databases using `SQLDatabase` and `SQLTableRetrieverQueryEngine`. This enables RAG over relational data. This feature is not in the original notebook.

In [ ]:
# New SQL Database Integration
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, SQLDatabase
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings
from llama_index.core.query_engine import SQLTableRetrieverQueryEngine
from sqlalchemy import create_engine, MetaData, Table, Column, Integer, String

# Set up LLM and embeddings
Settings.llm = Ollama(model="llama3")
Settings.embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Create a sample SQLite database
engine = create_engine("sqlite:///example.db")
metadata = MetaData()
products = Table(
    "products",
    metadata,
    Column("id", Integer, primary_key=True),
    Column("name", String),
    Column("description", String)
)
metadata.create_all(engine)

# Insert sample data
with engine.connect() as conn:
    conn.execute(products.insert(), [
        {"name": "Laptop", "description": "High-performance laptop for AI development"},
        {"name": "Smartphone", "description": "Latest model with AI assistant"}
    ])
    conn.commit()

# Create SQL database object
sql_database = SQLDatabase(engine, include_tables=["products"])

# Create SQL query engine
# Theory: SQLTableRetrieverQueryEngine translates natural language queries into SQL.
query_engine = SQLTableRetrieverQueryEngine(
    sql_database,
    llm=Settings.llm
)

# Query
response = query_engine.query("What products are available for AI development?")
print("Answer:", response)

## 13. Fine-Tuning Embeddings

**Theory**: Fine-tuning embeddings improves retrieval accuracy for domain-specific datasets. LlamaIndex supports this via integration with libraries like `sentence-transformers`. This feature is not in the original notebook.

In [ ]:
# New Embedding Fine-Tuning
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings
from sentence_transformers import SentenceTransformer, util
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
from sentence_transformers.trainer import SentenceTransformerTrainer
from datasets import Dataset

# Set up LLM and embeddings
Settings.llm = Ollama(model="llama3")
Settings.embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Load documents
documents = SimpleDirectoryReader(input_dir="data/").load_data()

# Prepare training data (simulated question-answer pairs)
qa_pairs = [
    {"question": "What is LlamaIndex?", "answer": documents[0].text},
    {"question": "What does LlamaIndex support?", "answer": documents[1].text}
]
dataset = Dataset.from_dict({
    "anchor": [pair["question"] for pair in qa_pairs],
    "positive": [pair["answer"] for pair in qa_pairs]
})

# Initialize model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Define training arguments
args = SentenceTransformerTrainingArguments(
    output_dir="./finetuned_model",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    learning_rate=2e-5
)

# Train model
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=dataset
)
trainer.train()

# Save fine-tuned model
model.save_pretrained("./finetuned_model")

# Update Settings with fine-tuned embeddings
Settings.embed_model = HuggingFaceEmbedding(model_name="./finetuned_model")

# Create index with fine-tuned embeddings
index = VectorStoreIndex.from_documents(documents)

# Create query engine
query_engine = index.as_query_engine(similarity_top_k=2)

# Query
response = query_engine.query("What is LlamaIndex?")
print("Answer:", response)

## Conclusion

This notebook covers the full spectrum of LlamaIndex functionalities, from basic indexing and querying to advanced features like document summary indexes, custom retrievers, sub-question query engines, SQL integration, and embedding fine-tuning. It preserves the original code while adding theoretical context and new examples to demonstrate LlamaIndex's versatility.

For further exploration, refer to the [LlamaIndex Documentation](https://docs.llamaindex.ai/) and experiment with different LLMs, embeddings, and data sources.